In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [33]:
#pip install trino pandas openpyxl openai

from trino.dbapi import connect
from trino.auth import BasicAuthentication
import pandas as pd
import json

In [ ]:
out_dir = r"C:\Users\namtv40\Data\DQ-Bitu-Silver-Data\schema"
# ===== Trino config =====

# --- 1. CẤU HÌNH HỆ THỐNG ---
CONFIG = {
    'host': os.getenv("VCS_TRINO_HOST"),
    'port': int(os.getenv("VCS_TRINO_PORT")),
    'user': os.getenv("VCS_TRINO_USER"),
    'catalog': 'hive',
    'schema': 'bitu_silver_data',  # Schema mục tiêu
    'password': os.getenv("VCS_TRINO_PASSWORD"),
}

trino_catalog = CONFIG['catalog']
trino_schema = CONFIG['schema']

def get_connection():
    return connect(
        host=CONFIG['host'],
        port=CONFIG['port'],
        user=CONFIG['user'],
        catalog=CONFIG['catalog'],
        http_scheme='http',
        auth=BasicAuthentication(CONFIG['user'], CONFIG['password']),
    )
    
def fetch_all(cursor, sql):
    cursor.execute(sql)
    return cursor.fetchall()
    

In [35]:
def load_table_metadata(table_name: str) -> dict:
    conn = get_connection()
    cursor = conn.cursor()

    # Table comment
    cursor.execute(f"""
        SELECT comment
        FROM system.metadata.table_comments
        WHERE catalog_name = '{trino_catalog}'
          AND schema_name = '{trino_schema}'
          AND table_name = '{table_name}'
        union  
        select description_vi  from hive.bitu_schema.table_business_convention  where db_name='{trino_schema}' and  table_name='{table_name}'
    """)
    table_comment = cursor.fetchall()
    table_comment = [item[0] for item in table_comment if item and item[0]]
    table_comment = "\n".join(table_comment)

    # Columns
    cursor.execute(f"""
        SELECT
    a1.column_name,
    a1.data_type,
    a1.is_nullable,
    (
        SELECT listagg(description_vi, ', ')
               WITHIN GROUP (ORDER BY description_vi)
        FROM {trino_catalog}.bitu_schema.column_business_convention b
        WHERE b.field_name = a1.column_name
          AND b.table_name = a1.table_name
    ) AS comment
FROM hive.information_schema.columns a1
WHERE a1.table_schema = '{trino_schema}'
  AND a1.table_name = '{table_name}'
ORDER BY a1.ordinal_position
    """)
    columns = []
    
    for row in cursor.fetchall():
        sql = f"""
        SELECT (CASE WHEN COUNT(DISTINCT {row[0]}) < 20
            THEN array_join(array_agg(DISTINCT CAST( {row[0]} AS VARCHAR)), ', ')
        ELSE NULL END ) as example
        FROM {trino_catalog}.{trino_schema}.{table_name} b
        """
        
        examples = fetch_all(cursor,sql=sql)
        examples = [str(r[0]) for r in examples if r and r[0]]
        examples = ', '.join(examples)
        
        columns.append({
            "column_name": row[0],
            "data_type": row[1],
            "nullable": row[2] == "YES",
            "sample_value": examples,
            "comment": row[3] or ""
        })
        
    conn.close()

    return {
        "database_name": trino_catalog,
        "schema_name": trino_schema,
        "table_name": table_name,
        "table_comment": table_comment or "",
        "columns": columns
    }

In [36]:
json_output = {
  "table_summary": {
    "database": "",
    "schema": "",
    "table_name": "",
    "business_description": "",
    "table_type": "",
    "confidence_level": ""
  },
  "columns": [
    {
      "column_name": "",
      "vietnamese_name": "",
      "business_description": "",
      "data_type": "",
      "nullable": True,
      "data_role": "",
      "is_primary_key": True,
      "is_foreign_key": True,
      "can_filter": True,
      "can_group": True,
      "is_metric": True,
      "related_table": "",
      "confidence_level": "",
      "notes": ""
    }
  ],
  "relationships": [
    {
      "column_name": "",
      "references_table": "",
      "reference_type": "",
      "assumption": "",
      "confidence_level": ""
    }
  ],
  "assumptions": [
    ""
  ],
  "quality_warnings": [
    ""
  ]
}

json_output_text = json.dumps(json_output, ensure_ascii=False)

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("VCS_LLM_API_KEY"),
    base_url=os.getenv("VCS_LLM_API_KEY")
)

def build_prompt(metadata: dict) -> str:
    database_name = metadata.get("database_name")
    schema_name = metadata.get("schema_name")
    table_name = metadata.get("table_name")
    table_comment = metadata.get("table_comment")
    list_of_columns = metadata.get("columns")
    column_text = ""
    for item in list_of_columns:
        column_name= item.get("column_name")
        data_type= item.get("data_type")
        nullable= item.get("nullable")
        sample_value= item.get("sample_value")
        comment= item.get("comment")
        column_text+= f"-{column_name} ({data_type}, nullable={nullable}): {comment}. Some sample values: {sample_value}\n"
        
    list_of_columns = json.dumps(metadata.get("columns"), ensure_ascii=False)
    return f"""
You are a senior Data Architect & Analytics Engineer.

## You specialize in:
- Data warehouse modeling
- Business data interpretation
- Metadata documentation
- SQL & BI semantic layers

Your task is to analyze table and column metadata and generate
clear, concise, business-friendly descriptions.

## Rules:
- Use Vietnamese
- Be precise, avoid hallucination
- If unsure, mark as "có thể là / giả định"
- Prioritize business meaning over technical detail
- Follow output schema exactly

# STRICT RULES (MANDATORY):
- Output MUST be valid JSON
- DO NOT include markdown, comments, explanations, or extra text
- Use Vietnamese language
- No trailing commas
- No null keys (use empty string "" or empty array [])
- If unsure, state assumption clearly
- Avoid hallucination
- Be concise, business-oriented

Dưới đây là metadata của một bảng trong hệ thống dữ liệu.

## Thông tin bảng:
- Database: {database_name}
- Schema: {schema_name}
- Table name: {table_name}
- Table comment (nếu có): {table_comment}

## Danh sách cột:
{column_text}

## Trong đó mỗi cột có cấu trúc:
- column_name
- data_type
- nullable
- sample_value (nếu có)
- comment (nếu có)

## Yêu cầu/Tasks:
1. Tóm tắt ý nghĩa nghiệp vụ của bảng (1–3 câu)
2. Với mỗi cột:
   - Diễn giải tên cột sang tiếng Việt dễ hiểu
   - Mô tả ý nghĩa nghiệp vụ
   - Suy đoán vai trò kỹ thuật nếu có:
     (primary_key, foreign_key, metric, dimension, timestamp, status, enum, identifier, other)
- Cột nào nên dùng để filter
- Cột nào nên dùng để group
- Cột nào là metric tính toán
3. Nếu phát hiện bảng thuộc loại:
   - Fact / Dimension / Bridge / Snapshot  / snapshot / transactional / unknown → hãy nêu rõ
4. Nếu suy luận được:
   - Mối quan hệ logic với bảng khác
5. Ghi rõ giả định & mức độ tin cậy 
- Ghi rõ mức độ tin cậy (cao / trung bình / thấp)
- Không được khẳng định tuyệt đối

### Định dạng output (BẮT BUỘC):
{json_output_text}

## Return JSON using EXACT schema previously defined.
"""

def analyze_with_llm(prompt: str) -> dict:
    response = client.chat.completions.create(
        model=None,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    return json.loads(content)

In [ ]:




def save_to_excel(result: dict, output_path: str):
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        # Table summary
        pd.DataFrame([result["table_summary"]]).to_excel(
            writer, sheet_name="table_summary", index=False
        )

        # Columns
        pd.DataFrame(result["columns"]).to_excel(
            writer, sheet_name="columns", index=False
        )

        # Relationships
        pd.DataFrame(result.get("relationships", [])).to_excel(
            writer, sheet_name="relationships", index=False
        )

        # Assumptions & warnings
        pd.DataFrame({
            "assumptions": result.get("assumptions", []),
            "quality_warnings": result.get("quality_warnings", [])
        }).to_excel(
            writer, sheet_name="notes", index=False
        )

def run(table_name: str, output_excel: str):
    metadata = load_table_metadata(table_name)
    print("\n\n====================metadata")
    print(metadata)
    prompt = build_prompt(metadata)
    print("\n\n====================prompt")
    print(prompt)
    result = analyze_with_llm(prompt)
    print("\n\n====================result")
    print(result)
    with open(output_excel +f"/{table_name}_lines.json", mode="w", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False, indent=4)+"\n")
    # save_to_excel(result, output_excel+"/"+table_name)



In [39]:
tables = [
    "deals",
"deals_allocation",
"deals_allocation_test",
"kinhdoanh",
"payment_records",
"plan_fin",
"plan_spdv_kd",
"pricebook",
"revenue",
"sales_accounts",
]
for table in tables:
    run(table, out_dir)



====================metadata
{'database_name': 'hive', 'schema_name': 'bitu_silver_data', 'table_name': 'deals', 'table_comment': 'Những cơ hội bán hàng dự kiến \u200b\u200btừ một khách hàng cụ thể (nằm trong bảng sales_accounts)', 'columns': [{'column_name': 'id', 'data_type': 'varchar', 'nullable': True, 'sample_value': '', 'comment': 'ID của cơ hội (pkey)'}, {'column_name': 'name', 'data_type': 'varchar', 'nullable': True, 'sample_value': '', 'comment': 'Tên cơ hội'}, {'column_name': 'amount', 'data_type': 'decimal(18,2)', 'nullable': True, 'sample_value': '', 'comment': 'Giá trị cơ hội tính bằng VND'}, {'column_name': 'base_currency_amount', 'data_type': 'decimal(18,2)', 'nullable': True, 'sample_value': '', 'comment': 'Giá trị cơ hội theo đơn vị tiền tệ tại thị trường'}, {'column_name': 'change_rate', 'data_type': 'double', 'nullable': True, 'sample_value': '1.0E0, 2.439024E4, 2.5E4, 2.272727E4, 2.564103E4, 2.631579E4', 'comment': ''}, {'column_name': 'expected_close_date', 'dat

In [40]:

# from collections import defaultdict

# def build_join_graph(analyzed_tables: list) -> dict:
#     """
#     analyzed_tables: list of LLM JSON outputs (nhiều bảng)
#     """
#     graph = defaultdict(list)

#     for table in analyzed_tables:
#         src_table = table["table_summary"]["table_name"]

#         for rel in table.get("relationships", []):
#             if rel["references_table"]:
#                 graph[src_table].append({
#                     "target_table": rel["references_table"],
#                     "on": f"{src_table}.{rel['column_name']} = "
#                           f"{rel['references_table']}.{rel['references_column']}",
#                     "join_type": rel.get("join_type", "LEFT JOIN"),
#                     "confidence": rel.get("confidence_level", "")
#                 })
#     return dict(graph)


# def generate_sql_template(table_analysis: dict) -> str:
#     table = table_analysis["table_summary"]["table_name"]
#     hints = table_analysis["sql_hints"]

#     select_metrics = ",\n  ".join(hints.get("metrics", ["COUNT(*)"]))
#     group_by = ", ".join(hints.get("default_group_by", []))
#     where_clause = " AND ".join(hints.get("default_where", ["1=1"]))

#     sql = f"""
# SELECT
#   {select_metrics}
# FROM {table}
# WHERE {where_clause}
# """

#     if group_by:
#         sql += f"\nGROUP BY {group_by}"

#     return sql.strip()


# def generate_semantic_layer(table_analysis: dict) -> dict:
#     semantic = table_analysis["semantic_layer"]

#     return {
#         "model": table_analysis["table_summary"]["table_name"],
#         "dimensions": [
#             {
#                 "name": dim,
#                 "type": "string"
#             } for dim in semantic.get("dimensions", [])
#         ],
#         "time_dimensions": [
#             {
#                 "name": td,
#                 "type": "time",
#                 "granularities": ["day", "month", "year"]
#             } for td in semantic.get("time_dimensions", [])
#         ],
#         "metrics": [
#             {
#                 "name": m["name"],
#                 "expression": m["expression"],
#                 "type": m.get("type", "sum")
#             } for m in semantic.get("metrics", [])
#         ],
#         "filters": semantic.get("filters", [])
#     }


# def save_join_graph_excel(join_graph: dict, path: str):
#     rows = []
#     for src, edges in join_graph.items():
#         for e in edges:
#             rows.append({
#                 "from_table": src,
#                 "to_table": e["target_table"],
#                 "join_condition": e["on"],
#                 "join_type": e["join_type"],
#                 "confidence": e["confidence"]
#             })

#     pd.DataFrame(rows).to_excel(path, index=False)


In [41]:
# run(
#         table_name="fact_orders",
#         output_excel="fact_orders_metadata.xlsx"
#     )